# Sparse Router / Lifecycle Calibration (Colab)


In [ ]:
import os, pathlib
repo_root = pathlib.Path('/content/That-Ai-Coder')
assert repo_root.exists(), 'Place repository at /content/That-Ai-Coder'
os.chdir(repo_root)


In [ ]:
from sparse_autosec.system import SparseExpertAutoSec
from sparse_autosec.config import AutoSecConfig

cfg = AutoSecConfig()
cfg.policy.max_fuzz_cases = 12
system = SparseExpertAutoSec(cfg)

tasks = [
    'eval injection in parser',
    'pickle deserialization endpoint',
    'shell true subprocess command'
]
for i in range(45):
    task = tasks[i % len(tasks)]
    complexity = system.core.score_task_complexity(task)
    decision = system.router.route(system.core.encode_task(task), complexity=complexity)
    sig = 'py_eval_user_input' if 'eval' in task else ('unsafe_deserialization' if 'pickle' in task else 'shell_injection')
    system.memory.counters[sig] = system.memory.counters.get(sig, 0) + 1
    reward = 1.0 if i % 3 != 0 else 0.0
    system.router.update_reward(decision.selected, reward)
    system.learning.record_replay(task, decision.selected[0], reward)

life = system.lifecycle.step()
print('spawned', life.spawned)
print('quiesced', life.quiesced)
print('retired', life.retired)


In [ ]:
for name in system.experts.names(active_only=False):
    ex = system.experts.get(name)
    print(name, ex.state.value, round(ex.health(), 4), ex.use_count, ex.success_count, ex.fail_count)
